In [70]:
import numpy as np
import math
import csv
import os
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.optimize import fsolve, differential_evolution,minimize

pi = np.pi


In [71]:
def print_row_parameters(row_index, csv_path="params_opt_value.csv"):
    """Finds a row by its index number and prints specific parameters."""
    try:
        with open(csv_path, mode="r", newline="", encoding="utf-8") as file:
            # DictReader automatically uses the first row (header) as keys
            reader = csv.DictReader(file)

            for current_index, row in enumerate(reader):
                if current_index == row_index:
                    # Print the specific formatting you requested
                    print(f"Log n = {row.get('Log n')}")
                    print(f"ZETA = {row.get('Zeta')}")
                    print(f"DELTA = {row.get('Delta')}")
                    print(f"OMEGA = {row.get('Omega')}")
                    print(f"k_param = {row.get('K')}")
                    print(f"TAU = {row.get('Tau')}")
                    print(f"ETA = {row.get('Eta')}")
                    print(f"THRESHOLD_M = {row.get('Threshold M')}")
                    return  # Exit the function once found

            print(f"Error: Row index {row_index} is out of bounds.")

    except FileNotFoundError:
        print(f"Error: The file at '{csv_path}' was not found.")

In [72]:
def bound_on_Dx(m1, Omega,Tau):
    return m1 * np.exp(-np.power(m1,1-Tau)/ (3 * np.log(m1) ** Omega))

def bound_on_Fx(value, Zeta, Log_space=False):
    if Log_space:
        M1 = m_func(value, Zeta, Log_space)
        return M1 * np.exp(-np.exp(value * (1 - Zeta)) / 4)
    else:
        return m_func(value, Zeta) * np.exp(-np.power(value, 1 - Zeta) / 4)

def bound_on_Tx(m1, K):
    return np.power(m1, -K / 6 + 2)

In [73]:
def zeta_bound(Epsilon):
    return 1 / (1 - Epsilon) * 3 / 4


def epsilon_bound(Zeta):
    return 1 - 3 / (4 * Zeta)

def optimal_epsilon(Zeta):
    return 1 / 2 - 3 / (8 * Zeta)


In [74]:
def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    # np.log2 works on floats; if using arrays, use np.log2
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def m_func(value,Zeta, Log_space=False):
    if Log_space:
        return np.ceil(np.exp(value*Zeta))
    else:
        return np.ceil(np.power(value,Zeta))

def C_epsilon(Epsilon): #H(ε) + H(ε')(1-ε) + ˜ε log ˜ε − ˜ε
    Epsilon_prime = Epsilon/(1-Epsilon)
    return binary_entropy(Epsilon) + binary_entropy(Epsilon_prime)*(1-Epsilon) + Epsilon*np.log(Epsilon) - Epsilon




In [75]:
def exps_bound_of_Ee(value,Epsilon,Delta,Zeta,c0):
    Epsilon_prime = Epsilon/(1-Epsilon)
    L_term = 3*Epsilon + 4*Zeta*(Delta-Epsilon) -2*Delta
    #linear term is 3ε + 4ζ(δ − ε) -2δ
    log_alpha = Delta*np.log(3/2)+(Epsilon-Delta)*np.log(3*Delta)
    constant_term = 2*Delta + 2*log_alpha + C_epsilon(Epsilon)

    #C = 2πδc = (2π)^3/2 δε sqrt[(1 − ε)(1 − ε′)].
    C = np.pow(2*pi,3/2)* Delta*Epsilon*np.sqrt((1-Epsilon)*(1-Epsilon_prime))
    exp_term = -3*value/2 -np.log(C)
    result = constant_term + value*L_term + np.exp(-value)*exp_term #+ 1/(12*Epsilon)*np.exp(-2*value)
    #this result is then multiplied by n, and exponentiated, so value becomes negligible
    # print(L_term,constant_term,exp_term,result)
    return result

In [76]:
def A_condition(value,Zeta,Epsilon,Gamma,Omega,Tau,K,c0,Eta):
    C_k = 2*np.exp(2)*(4*K+2)*(2*K+1) #+ 2*np.exp(2)
    # C_k = 2*np.exp(2)*(4*K)*(2*K)
    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda = Epsilon*c0
    log_M = Zeta*value
    log_M_omega = np.power(log_M,-Omega)
    Term1 = Gamma/(C_k *np.power(M1,2*Tau) * np.power(log_M,2*Eta)) * np.log(1-log_M_omega**2 * np.power(1-log_M_omega,4*K*log_M**Eta))
    #coupled with e^L

    # Term1 = -Gamma/(C_k*np.power(M1,2*Tau)*np.power(log_M,2*Eta+2*Omega)) * np.power(1-1/log_M**Omega,4*K*log_M**Eta)

    Term2 = C_epsilon(Epsilon)
    #constant term

    # c=ε sqrt[2π(1 − ε)(1 − ε′)].
    Epsilon_prime = Epsilon/(1-Epsilon)
    c = Epsilon*np.sqrt(2*pi*(1-Epsilon)*(1-Epsilon_prime))
    Term3 = -value/2 - np.log(c)
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return Term1*np.exp(value) +Term2+Term3*np.exp(-value)+ Term4*value #+ 1/(12*Epsilon)*np.exp(-2*value)

def H_condition(value,Zeta,Gamma,Epsilon,Tau,c0,Constant1=3/(52*np.exp(2))):#constant 1 is the value that arises from lemma 7.2

    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda=Epsilon*c0
    Term1 = Constant1*Gamma/np.power(M1,2*Tau)
    #coupled with e^L
    Term2 = C_epsilon(Epsilon)
    #constant
    Epsilon_prime = Epsilon/(1-Epsilon)
    c = Epsilon*np.sqrt(2*pi*(1-Epsilon)*(1-Epsilon_prime))
    Term3 = -value/2 - np.log(c)
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return -Term1*np.exp(value) +Term2+Term3*np.exp(-value)+Term4*value #+ 1/(12*Epsilon)*np.exp(-2*value)


In [77]:
def gamma_n(value,Zeta,Delta,Omega,Tau,Log_space=True):
    if Log_space:
        n_value = np.exp(value)
        M1 = m_func(value,Zeta,Log_space=Log_space)
        return Delta**2/4 - Delta/(4*n_value) - 16*np.power(M1,1-2*Tau)/np.power(np.log(M1),2*Omega)

# def thm31_req(value,Zeta,Delta, Omega, Tau, bound,Log_space=True):
#     if Log_space:
#         M1 = m_func(value,Zeta,Log_space=Log_space)
#         return bound > 4/M1 + 16*np.power(M1,1-2*Tau)/np.power(np.log(M1),2*Omega) + Delta/(4*np.exp(value))
#
# def lemma42_req(value,Zeta,Epsilon,Log_space=True):
#     M1 = m_func(value, Zeta, Log_space)
#     if Log_space:
#         return 2 + M1  <= Epsilon**2*np.exp(value)

In [78]:
def log_M_condition(log_M, threshold_for_m, Eta, K,Tau): #threshold is NEGATIVE
    # if threshold_for_m >= 0:
    #     return False


    log_M_eta = np.pow(log_M,Eta-1)
    m_tau = np.exp(log_M*(1-2*Tau))# - 2/np.exp(log_M*2*Tau)
    # print(f"m^(1-2τ) = {m_tau}")
    # Using np.log and python's native ** power operator
    terms= [
            2 - m_tau/log_M,
            K*log_M_eta,
            -K*np.log(K)*log_M_eta,
            -K*Eta*log_M_eta*np.log(log_M),
            # K*log_M_eta*np.log(m_tau),
            K*log_M_eta*(1-2*Tau)*log_M,
        ]
    # print(term1+term2-term3)
    return sum(terms)*log_M < -0.7

In [79]:
# def least_L_with_params(zeta, delta, gamma, omega, k, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):
def least_L_with_params(zeta, delta, omega, k, eta, threshold_M=-0.7, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):

    # Assuming epsilon and zeta_bound are defined
    params = {
        "zeta": zeta,
        "delta": delta,
        # "gamma": gamma,
        "k": k
    }

    # Define requirements as (Current Value, Boolean Condition, Description)
    # tau_lower_bound=0.4
    requirements = {
        # "Tau": (tau,1>tau>=tau_lower_bound, f"{tau_lower_bound} <= τ < 1" ),
        "Zeta": (zeta, 1 > zeta > 2 / (2 + tau), f"{2 / (2 + tau):.6f} < ζ < 1"),
        # "Delta": (delta, 0< delta < epsilon*(1 - c0/(4*zeta-2)), f" δ < {epsilon*epsilon*(1 - (2+c0)/(4*zeta)):.3e}" ),
        # "Gamma": (gamma, 0 < gamma < delta ** 2 / 4, f"0 < γ < {delta**2 / 4:.2e}"),
        # "k": (k, k >12 , "k > 12")
    }

    # Check for failures
    failed = [name for name, (val, met, desc) in requirements.items() if not met]

    if failed:
        # print("═══ PARAMETER CHECK FAILED ═══")
        # for name, (val, met, desc) in requirements.items():
        #     status = "✓" if met else "✗"
        #     print(f"{status} {name:6} : {val:<8} | Required: {desc}")

        return 1e12
        # raise ValueError(f"Constraint violation in: {', '.join(failed)}")

    # print("✓ All parameters satisfy requirements.")

    def log_M_feasible(L):
        M2 = m_func(value=L, Zeta=zeta, Log_space=True)
        log_M = np.log(M2)
        return log_M_condition(log_M, threshold_M, eta, k,tau)

    # Define a helper to check all conditions for a given L
    def is_feasible(L):
        # Local evaluation space variables

        M = m_func(value=L, Zeta=zeta, Log_space=True)
        log_M = np.log(M)
        # New Triangle Parameter Condition based on log^\eta(m) instead of k*log(m)
        # Replacing the loose linear Chernoff/McDiarmid threshold bounds

        gamma_opt = gamma_n(value=L, Zeta=zeta, Delta=delta, Omega=omega, Tau=tau, Log_space=True)
        H_exp = H_condition(value=L, Zeta=zeta, Gamma=gamma_opt, Tau=tau, Epsilon=epsilon, c0=c0)

        # Injects the sharper non-linear triangle concentration criteria into Event A
        A_exp = A_condition(value=L, Zeta=zeta, Epsilon=epsilon, Gamma=gamma_opt,
                                      K=k, Tau=tau, Omega=omega, c0=c0, Eta=eta)

        threshold = -1e-60
        Fx_bound = bound_on_Fx(value=L, Zeta=zeta, Log_space=True)
        Dx_bound = bound_on_Dx(M, omega, tau)
        Ee_bound = exps_bound_of_Ee(value=L, Epsilon=epsilon, Delta=delta, Zeta=zeta, c0=c0)

        conds = [
            Ee_bound < threshold,
            H_exp < threshold,
            A_exp < threshold,
            # np.exp(L) * delta > M,
            M-2 <= np.pow(M,2*tau)*k*np.pow(log_M,eta),
            1 > zeta > (2+np.log(4)/L) / (2 + tau)
            # Explicit non-linear matching check
        ]

        # Sum of early error bounds must strictly clear union criteria mass
        # print(f"{1.0-2*np.exp(-1*log_M) - np.exp(A_exp*np.exp(L)) - np.exp(H_exp*np.exp(L)):.2e}")
        return 2 * (Fx_bound + Dx_bound) < 0.999-2*np.exp(-0.7) and all(conds),conds

    # --- DYNAMIC RANGE RESOLUTION ---
    # Since it's no longer a uniform net cut, find the window boundaries manually
    low = 100
    high = 700



    # Initialize boundaries for the binary/logarithmic search
    search_low = low
    search_high = high
    adjusted_high = None

    # A tolerance threshold determines how precise you want the boundary to be.
    # 1e-5 is usually plenty, but you can adjust it based on your precision needs.

    while (search_high - search_low) > 1e-10:
        # Find the midpoint of the current window
        test_high = (search_low + search_high) / 2.0

        if log_M_feasible(test_high):
            # This point is safe! Save it, and try to push higher.
            adjusted_high = test_high
            search_low = test_high
        else:
            # Asymptotic flip hit. We need to look lower.
            search_high = test_high

    if adjusted_high is None:
        return 1e6  # Interval completely broken or fundamentally unfeasible

    # print(f"L can be at most {adjusted_high}")


    # Reset high boundary safely to the upper limits of the verified active zone
    high = adjusted_high
    best_L = high


    feasible,condss = is_feasible(high)
    if not feasible:
        # print(condss)
        return 1e9


    # for L in np.linspace(search_low, search_high, 10*(search_high-search_low+1)):
    #     feasible,_ = is_feasible(L)
    #     if feasible:
    #         best_L = L
    #         break
    while (high - low) > tol:
        mid = (low + high) / 2
        # print(f"Mid is ={mid}")
        feasible,_ = is_feasible(mid)
        if feasible:
            # print(f"Passes here")
            best_L = mid
            high = mid  # Try to find a smaller L in the lower half
        else:
            low = mid   # Need a larger L, look in the upper half

    # gamma_opt = gamma_n(value=best_L-1, Zeta=zeta, Delta=delta, Omega=omega, Tau=tau, Log_space=True)
    # print(f"bottle neck is exp_A: {not A_condition(value=best_L-1, Zeta=zeta, Epsilon=epsilon, Gamma=gamma_opt,K=k, Tau=tau, Omega=omega, c0=c0, Eta=eta)<-1e-50}")
    # _,conds_before = is_feasible(best_L-0.4)
    # print(conds_before)
    # print(condss)
    return best_L

In [80]:
# ZETA= 8.00000000000001043610e-01
# DELTA = 1.549369e-02
# # GAMMA= 5.943461e-05
# OMEGA= 1.6
# k_param=1.201872e+01
# least_L_with_params(zeta=ZETA,delta=DELTA,omega=OMEGA,k=k_param,eta=0.78,c0=1+1e-30,threshold_M=-1)


In [81]:
def objective(params):
    # 1. Unpack exactly matching the order of x0
    zeta, delta, omega, k, tau, eta = params
    # zeta, delta, omega, k, tau,  threshold_m = params
    # 2. Pass EVERYTHING explicitly by name to eliminate positional shifting
    L = least_L_with_params(zeta=zeta,delta=delta,omega=omega,k=k,eta=eta,tau=tau,epsilon=0.25,c0=1,tol=1e-9)
    return L
ZETA= 8.00765277138820730229e-01
DELTA = 4.266966026579865e-02
OMEGA= 1.003208169939821
k_param=28.553955017639169
TAU=0.506573195106921
ETA=1.485056097386873e-01
# THRESHOLD_M=-0.01035
# print(ZETA > 2/(2+TAU))
# Bounds for (zeta, epsilon, gamma, omega, k)
# Using small buffers (1e-6) to avoid exact boundary issues
bounds = [
    (0.75, 0.9999),  # zeta: 0.8 < zeta < 1
    #(0.0001, 0.25),  # epsilon: typically small positive
    (0.0001,0.25), #delta: small positive
    # (1e-12, 0.1),  # gamma: 0 < gamma < epsilon**4 / 4
    (0.5001, 10.0),  # omega: typically > 1
    (0.001, 50000.0),  # k: k > e
    (0.45,0.99999), # tau: 0.5=< tau < 1
    (0,1), #eta <= 1
    # (-5,-0.00001)#Threshold:M
]
# Initial guess (starting point for the optimizer)
x0 = [ZETA, DELTA, OMEGA, k_param, TAU, ETA]
# x0 = [ZETA, DELTA, OMEGA, k_param, TAU, THRESHOLD_M]

# We use COBYLA or Nelder-Mead because your function is likely not differentiable
res = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    bounds=bounds,
    options={'maxiter': 5000, 'disp': True}
)

if res.success:
    # optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau,best_threshold = res.x
    optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau,optimized_eta = res.x
    # optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau,optimized_eta,best_threshold = res.x
    my_result = res.fun


    # Check if it actually found a valid structural constraint index
    if my_result < 1000.0:
        print(f"Minimum L found: {my_result:10f}")
        print(f"Value of n : {np.exp(res.fun):.6e}")
    else:
        print("Optimization 'converged' but only found invalid/penalized regions.")

    print(f"ZETA= {optimized_zeta:.20e}")
    # print(f"EPSILON= {optimized_epsilon:.6e}")
    print(f"DELTA = {optimized_delta:.15e}" )
    # print(f"GAMMA= {optimized_gamma:.15e}")
    print(f"OMEGA= {optimized_omega:.15f}")
    print(f"k_param={optimized_k:.15f}")
    print(f"TAU={optimized_tau:.15f}")
    print(f"ETA={optimized_eta:.15e}")
    # print(f"THRESHOLD_M={best_threshold:.5f}")


else:
    print("Optimization failed to converge.")

Optimization terminated successfully.
         Current function value: 193.162111
         Iterations: 539
         Function evaluations: 963
Minimum L found: 193.162111
Value of n : 7.748881e+83
ZETA= 8.00765329394364844262e-01
DELTA = 4.266965010716672e-02
OMEGA= 1.003208034347382
k_param=28.818098585380611
TAU=0.506573128671516
ETA=1.466791089324265e-01


In [82]:
ZETA= 8.00890133204898324593e-01
DELTA = 3.947947591621143e-02
OMEGA= 1.017492180586969
k_param=35.121731148449300
TAU=0.506126097354749
ETA=1.129795043077065e-01
THRESHOLD_M=-0.01074

# print(DELTA < 0.25*(1-1/(4*ZETA-2)))
#
# print(exps_bound_of_Ee(140.732013343949802219867706298828125000000000000000000000000000,0.25,DELTA,ZETA,1))
log_M = ZETA*195.463920
print(log_M_condition(log_M,THRESHOLD_M,ETA,k_param,TAU))

SSS = least_L_with_params(
        zeta=ZETA,
        delta=DELTA,
        omega=OMEGA,
        k=k_param,
        tau=TAU,
        eta=ETA,
        threshold_M=THRESHOLD_M
    )
if SSS < 1000:
    print(f"{SSS:.60f}")
else:
    print(f"{SSS:.2e}")



# print(exps_bound_of_Ee(171.48925961427094,0.25,DELTA,ZETA,1))
# M = np.exp(ZETA*189.2612886818202468930394388735294342041015625)
# print(M-2 <= np.pow(M,2*TAU)*k_param*np.pow(np.log(M),ETA))

True
194.385822583969058996444800868630409240722656250000000000000000


In [83]:
ZETA= 8.00890133204898324593e-01
DELTA = 0.04539737713628135
OMEGA = 1.47032747393191
k_param = 84.31735845913083
TAU = 0.487904661147198
ETA = 0.007796201068497075
THRESHOLD_M = -0.00606
# print(ZETA > 2/(2+TAU))
result1 = least_L_with_params(
        zeta=ZETA,
        delta=DELTA,
        omega=OMEGA,
        k=k_param,
        tau=TAU,
        eta=ETA,
        threshold_M=THRESHOLD_M
    )

print(result1)
file_path = 'params_opt_value.csv'
data_row = [result1,ZETA,DELTA,OMEGA,k_param,TAU,ETA,THRESHOLD_M]
# with open('params_opt_value.csv', "a", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(data_row)
#     print(f"New data entry created in :{file_path}")
# print(ZETA > 2/(2+TAU))
# ZETA= 8.03999031283763265776e-01
# DELTA = 4.428224435653951e-02
# OMEGA= 1.482431350114056
# k_param=84.564312314334558
# TAU=0.487659717713236
# ETA=7.615236719605971e-03
# THRESHOLD_M=-0.00612

#best value found until here 189.8000200851068370866414625197649002075195

1000000000000.0


In [84]:
print(1/0.007796201068497075)

128.26759997773846


In [85]:
print_row_parameters(0)

Log n = 189.27659129310751
ZETA = 0.8038893261608455
DELTA = 0.04539737713628135
OMEGA = 1.47032747393191
k_param = 84.31735845913083
TAU = 0.487904661147198
ETA = 0.007796201068497075
THRESHOLD_M = -0.00606
